# Bipartite graph scratch

In [3]:
import numpy as np
import polars as pl
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import optuna
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt


In [7]:
rna = pl.read_csv("xena_brca_data/TCGA.BRCA.sampleMap_HiSeqV2", separator="\t")

In [8]:
gene_names = rna[:,0]
rna = rna.drop("sample")

In [9]:
# load labels

clinicalMatrix = pl.read_csv("xena_brca_data/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix", separator="\t", infer_schema_length=0)

cM = clinicalMatrix.filter(pl.col("PAM50_mRNA_nature2012") != "null")
cM


sampleID,AJCC_Stage_nature2012,Age_at_Initial_Pathologic_Diagnosis_nature2012,CN_Clusters_nature2012,Converted_Stage_nature2012,Days_to_Date_of_Last_Contact_nature2012,Days_to_date_of_Death_nature2012,ER_Status_nature2012,Gender_nature2012,HER2_Final_Status_nature2012,Integrated_Clusters_no_exp__nature2012,Integrated_Clusters_unsup_exp__nature2012,Integrated_Clusters_with_PAM50__nature2012,Metastasis_Coded_nature2012,Metastasis_nature2012,Node_Coded_nature2012,Node_nature2012,OS_Time_nature2012,OS_event_nature2012,PAM50Call_RNAseq,PAM50_mRNA_nature2012,PR_Status_nature2012,RPPA_Clusters_nature2012,SigClust_Intrinsic_mRNA_nature2012,SigClust_Unsupervised_mRNA_nature2012,Survival_Data_Form_nature2012,Tumor_T1_Coded_nature2012,Tumor_nature2012,Vital_Status_nature2012,_INTEGRATION,_PANCAN_CNA_PANCAN_K8,_PANCAN_Cluster_Cluster_PANCAN,_PANCAN_DNAMethyl_BRCA,_PANCAN_DNAMethyl_PANCAN,_PANCAN_RPPA_PANCAN_K8,_PANCAN_UNC_RNAseq_PANCAN_K16,_PANCAN_miRNA_PANCAN,…,progesterone_receptor_level_cell_percent_category,project_code,radiation_therapy,sample_type,sample_type_id,surgical_procedure_purpose_other_text,system_version,targeted_molecular_therapy,tissue_prospective_collection_indicator,tissue_retrospective_collection_indicator,tissue_source_site,tumor_tissue_site,vial_number,vital_status,year_of_initial_pathologic_diagnosis,_GENOMIC_ID_TCGA_BRCA_exp_HiSeqV2_exon,_GENOMIC_ID_TCGA_BRCA_exp_HiSeqV2_PANCAN,_GENOMIC_ID_TCGA_BRCA_RPPA_RBN,_GENOMIC_ID_TCGA_BRCA_mutation,_GENOMIC_ID_TCGA_BRCA_PDMRNAseq,_GENOMIC_ID_TCGA_BRCA_hMethyl450,_GENOMIC_ID_TCGA_BRCA_RPPA,_GENOMIC_ID_TCGA_BRCA_PDMRNAseqCNV,_GENOMIC_ID_TCGA_BRCA_mutation_curated_wustl_gene,_GENOMIC_ID_TCGA_BRCA_hMethyl27,_GENOMIC_ID_TCGA_BRCA_PDMarrayCNV,_GENOMIC_ID_TCGA_BRCA_miRNA_HiSeq,_GENOMIC_ID_TCGA_BRCA_mutation_wustl_gene,_GENOMIC_ID_TCGA_BRCA_miRNA_GA,_GENOMIC_ID_TCGA_BRCA_exp_HiSeqV2_percentile,_GENOMIC_ID_data/public/TCGA/BRCA/miRNA_GA_gene,_GENOMIC_ID_TCGA_BRCA_gistic2thd,_GENOMIC_ID_data/public/TCGA/BRCA/miRNA_HiSeq_gene,_GENOMIC_ID_TCGA_BRCA_G4502A_07_3,_GENOMIC_ID_TCGA_BRCA_exp_HiSeqV2,_GENOMIC_ID_TCGA_BRCA_gistic2,_GENOMIC_ID_TCGA_BRCA_PDMarray
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""TCGA-A1-A0SD-0…","""Stage IIA""","""59""","""2""","""Stage IIA""","""437""",null,"""Positive""","""FEMALE""","""Negative""",null,null,null,"""Negative""","""M0""","""Negative""","""N0""","""437""","""0""","""LumA""","""Luminal A""","""Positive""",null,"""-9""","""-3""","""enrollment""","""T_Other""","""T2""","""LIVING""","""TCGA-A1-A0SD-0…","""High""","""C3-BRCA/Lumina…","""cluster 1""","""BRCA non-CIMP …",null,"""BRCA nonbasal-…","""miRNA cluster …",…,"""90-99%""",null,null,"""Primary Tumor""","""01""",null,"""6th""",null,"""NO""","""YES""","""A1""","""Breast""","""A""","""LIVING""","""2005""","""15bad71d-3031-…","""15bad71d-3031-…",null,"""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…",null,null,"""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…",null,"""15bad71d-3031-…",null,"""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…","""15bad71d-3031-…","""TCGA-A1-A0SD-0…","""TCGA-A1-A0SD-0…"
"""TCGA-A1-A0SE-0…","""Stage I""","""56""","""2""","""Stage I""","""1320""",null,"""Positive""","""FEMALE""","""Negative""",null,null,null,"""Negative""","""M0""","""Negative""","""N0""","""1320""","""0""","""LumA""","""Luminal A""","""Positive""",null,"""-5""","""-7""","""enrollment""","""T1""","""T1""","""LIVING""","""TCGA-A1-A0SE-0…","""Iq""","""C3-BRCA/Lumina…","""cluster 1""","""BRCA non-CIMP …",null,"""BRCA nonbasal-…","""miRNA cluster …",…,"""90-99%""",null,null,"""Primary Tumor""","""01""",null,"""6th""",null,"""NO""","""YES""","""A1""","""Breast""",null,"""LIVING""","""2005""","""a998

In [10]:
labels = cM.select(["sampleID","PAM50_mRNA_nature2012"])
labels = labels.filter(pl.col("PAM50_mRNA_nature2012") != "Normal-like")

In [11]:
y_all = labels["PAM50_mRNA_nature2012"].to_numpy()
y_all

array(['Luminal A', 'Luminal A', 'Luminal A', 'Luminal A', 'Basal-like',
       'Luminal B', 'Basal-like', 'Luminal A', 'Basal-like', 'Basal-like',
       'Luminal B', 'Basal-like', 'Basal-like', 'Luminal A',
       'HER2-enriched', 'HER2-enriched', 'Luminal A', 'HER2-enriched',
       'Basal-like', 'Luminal A', 'Luminal A', 'Luminal A', 'Luminal B',
       'Luminal A', 'Luminal A', 'Luminal B', 'HER2-enriched',
       'HER2-enriched', 'Luminal A', 'Basal-like', 'HER2-enriched',
       'Basal-like', 'Luminal A', 'Luminal B', 'Luminal A', 'Luminal A',
       'Luminal A', 'HER2-enriched', 'Luminal B', 'Luminal A',
       'Luminal A', 'Luminal A', 'Luminal A', 'Luminal A', 'Luminal A',
       'Luminal B', 'Basal-like', 'Luminal A', 'Luminal B', 'Luminal B',
       'Basal-like', 'Luminal A', 'Basal-like', 'HER2-enriched',
       'Basal-like', 'Luminal B', 'Luminal B', 'Luminal A', 'Luminal A',
       'Luminal A', 'Luminal A', 'Luminal A', 'Basal-like', 'Luminal A',
       'Luminal B', 'Lum

In [12]:
np.unique(labels["PAM50_mRNA_nature2012"].to_numpy(), return_counts=True)

(array(['Basal-like', 'HER2-enriched', 'Luminal A', 'Luminal B'],
       dtype=object),
 array([ 98,  58, 231, 127]))

In [13]:
rna = rna.select(labels["sampleID"].to_numpy())

In [14]:
rna

TCGA-A1-A0SD-01,TCGA-A1-A0SE-01,TCGA-A1-A0SH-01,TCGA-A1-A0SJ-01,TCGA-A1-A0SK-01,TCGA-A1-A0SM-01,TCGA-A1-A0SO-01,TCGA-A2-A04N-01,TCGA-A2-A04P-01,TCGA-A2-A04Q-01,TCGA-A2-A04R-01,TCGA-A2-A04T-01,TCGA-A2-A04U-01,TCGA-A2-A04V-01,TCGA-A2-A04W-01,TCGA-A2-A04X-01,TCGA-A2-A04Y-01,TCGA-A2-A0CL-01,TCGA-A2-A0CM-01,TCGA-A2-A0CP-01,TCGA-A2-A0CQ-01,TCGA-A2-A0CS-01,TCGA-A2-A0CT-01,TCGA-A2-A0CU-01,TCGA-A2-A0CV-01,TCGA-A2-A0CW-01,TCGA-A2-A0CX-01,TCGA-A2-A0CY-01,TCGA-A2-A0CZ-01,TCGA-A2-A0D0-01,TCGA-A2-A0D1-01,TCGA-A2-A0D2-01,TCGA-A2-A0D3-01,TCGA-A2-A0D4-01,TCGA-A2-A0EM-01,TCGA-A2-A0EN-01,TCGA-A2-A0EO-01,…,TCGA-E2-A14W-01,TCGA-E2-A14X-01,TCGA-E2-A14Y-01,TCGA-E2-A14Z-01,TCGA-E2-A150-01,TCGA-E2-A152-01,TCGA-E2-A153-01,TCGA-E2-A154-01,TCGA-E2-A155-01,TCGA-E2-A156-01,TCGA-E2-A158-01,TCGA-E2-A159-01,TCGA-E2-A15A-01,TCGA-E2-A15C-01,TCGA-E2-A15D-01,TCGA-E2-A15E-01,TCGA-E2-A15F-01,TCGA-E2-A15G-01,TCGA-E2-A15H-01,TCGA-E2-A15I-01,TCGA-E2-A15J-01,TCGA-E2-A15K-01,TCGA-E2-A15L-01,TCGA-E2-A15M-01,TCGA-E2-A15O-01,TCGA-E2-A15P-01,TCGA-E2-A15R-01,TCGA-E2-A15S-01,TCGA-E2-A15T-01,TCGA-E2-A1AZ-01,TCGA-E2-A1B0-01,TCGA-E2-A1B1-01,TCGA-E2-A1B4-01,TCGA-E2-A1B5-01,TCGA-E2-A1B6-01,TCGA-E2-A1BC-01,TCGA-E2-A1BD-01
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
9.1657,9.7179,8.8669,9.7663,9.8632,8.8234,8.6701,9.9831,10.5102,10.064,9.5425,10.449,9.4763,10.5333,8.9774,8.9665,9.3774,9.5693,11.4132,9.6413,10.3112,9.2703,9.0876,11.2247,8.9437,9.741,8.9323,8.4062,10.1639,8.3209,11.1395,10.1611,9.2467,10.9353,9.3572,10.0545,9.778,…,8.9611,10.7636,9.9899,9.8046,11.2344,9.8236,9.984,9.2405,9.8415,10.3731,9.2199,10.1219,9.1705,10.0851,9.2262,9.7065,10.0334,10.0949,9.4848,9.1406,9.2531,9.9412,9.3713,8.453,9.376,9.1228,9.1934,10.0346,9.8209,9.1036,8.5746,9.3223,10.4367,8.9027,8.6965,9.0303,10.7991
2.4935,3.3983,1.6631,6.1765,7.497,1.8177,8.9699,1.7558,6.9166,2.2799,1.4911,9.2573,6.3214,2.9105,2.5516,1.6612,3.6294,3.6849,0.0,4.8982,0.4407,2.3655,1.1726,2.9769,3.0291,3.35,0.8538,5.0676,4.063,1.7954,1.9355,2.1132,0.0,4.0158,1.4728,4.2849,2.5901,…,1.7638,4.8435,2.8783,2.9498,2.9782,2.3097,1.6892,1.9935,2.1329,0.7806,4.3358,4.1235,0.7135,0.848,3.8032,4.4089,0.8836,2.4616,1.1422,3.0117,0.4366,1.3397,1.6929,4.9786,0.8672,1.7922,0.3056,5.0409,0.3936,5.7418,1.3134,4.919,1.8654,4.8873,4.6748,6.9578,2.1772
0.0,0.0,0.4449,0.0,0.7051,1.178,0.0,0.0,0.6028,0.0,0.0,0.0,0.0,2.0906,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4891,0.6906,0.0,7.7299,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.8064,0.0,0.0,0.0,0.0,0.638,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0372,0.0,0.0,1.4776,0.0,0.8114
11.7868,11.3219,11.8779,11.502,12.099,11.9934,11.6209,12.185,12.1741,11.7065,11.9362,11.3839,12.5415,12.22,12.0188,12.5742,12.0841,11.6492,12.0196,12.0527,12.2531,12.3055,11.1702,11.5801,12.0265,12.2589,11.945,12.9661,11.6858,11.8358,13.0469,11.7749,11.9473,12.1437,11.6602,12.0735,11.881,…,12.3585,11.8525,11.7697,12.3889,11.3972,12.1153,11.7244,11.5464,11.9198,12.0953,11.8904,12.0105,12.2024,11.9198,11.9175,11.7502,12.0135,12.2843,12.3655,12.0196,11.9345,11.9704,11.88,11.7094,12.4611,12.4242,11.7385,12.4475,11.8324,11.416,11.7854,11.7005,12.428,12.6904,12.3403,11.8271,12.2783
11.0131,11.2617,11.8526,11.4405,9.9933,11.9332,11.0278,11.3242,11.2632,10.7574,10.4421,10.6897,10.9603,11.2698,11.9258,11.0061,10.9864,11.0977,11.1327,11.1117,11.1687,10.7255,11.4965,11.116,11.5781,10.6494,10.7706,10.4751,11.2883,10.838,11.5712,10.9105,12.0452,10.1117,11.7259,10.9669,11.3562,…,11.3929,11.1539,11.0064,10.2678,11.758,11.8669,11.1408,10.3065,11.696,11.6226,10.6569,12.1943,10.6067,11.4468,11.066,11.5324,10.837,11.112,10.7207,11.3718,10.688,10.898,11.183,11.415,10.718,10.5873,10.1383,10.8017,11.1716,11.2437,10.9101,11.6778,11.0804,11.

In [15]:
# filter low var rows
rna_mat = rna.to_numpy()
rna_vars = rna_mat.var(axis=1)
high_var_mask = rna_vars > 0.01

np.unique(high_var_mask, return_counts=True)

high_var_idx = np.where(high_var_mask)[0]

In [16]:
gene_names = gene_names[high_var_idx]
rna = rna[high_var_idx]

print(rna.shape, gene_names.shape)

(19767, 514) (19767,)


In [17]:
label_encoder = LabelEncoder()
label_encoder.fit(y_all)
y_all_e = label_encoder.transform(y_all)

# add numberd labels to labels df
labels = labels.with_columns(y=y_all_e)

In [18]:
# labels and rna columns are aligned, do train, val, test split

# random split for train / test
train_temp_idx, test_idx = train_test_split(
    np.arange(rna.shape[1]),
    test_size=0.3,
    # stratify=y,
    random_state=4
)

In [19]:
train_idx, val_idx = train_test_split(
    train_temp_idx, 
    test_size=0.1, 
    stratify=y_all_e[train_temp_idx],                   
    random_state=4
)

In [20]:
# split dataframes
train_df = rna[:, train_idx.tolist()]
train_labels = labels[train_idx]
val_df = rna[:, val_idx.tolist()]
val_labels = labels[val_idx]
test_df = rna[:, test_idx.tolist()]
test_labels = labels[test_idx]

In [21]:
# select features based on the training set
num_labels = len(np.unique(y_all_e))
genes_class_mean = np.zeros((rna.shape[0], num_labels))

# separate training samples by class
for label in range(num_labels):
    # select cols which are from this class
    label_samples = train_labels.filter(pl.col("y") == label)["sampleID"].to_numpy()
    label_train_df = train_df[label_samples].to_numpy()

    genes_class_mean[:, label] = label_train_df.mean(axis=1)

In [22]:
genes_class_variance = genes_class_mean.var(axis=1)

class_var_order = np.argsort(genes_class_variance)

# select the 500 genes with the largest variance
best_500 = class_var_order >= (genes_class_variance.shape[0] - 500)
best_500_idx = np.where(best_500)[0]

# selection itself
train_df = train_df[best_500_idx]
val_df = val_df[best_500_idx]
test_df = test_df[best_500_idx]

In [23]:
X_train = train_df.to_numpy().T
y_train = train_labels["y"].to_numpy()
X_val = val_df.to_numpy().T
y_val = val_labels["y"].to_numpy()
X_test = test_df.to_numpy().T
y_test = test_labels["y"].to_numpy()

# scale everything
std_scale = StandardScaler().fit(X_train)
X_train = std_scale.transform(X_train) 
X_val = std_scale.transform(X_val)
X_test = std_scale.transform(X_test)

In [25]:
# eval using knns
ks = (1, 30)

# eval on rbf svms
def objective_knn(trial):

    knn = KNeighborsClassifier(
        n_neighbors=trial.suggest_int("n_neighbors", ks[0], ks[1]),
        weights=trial.suggest_categorical("weights", ["uniform", "distance"]),
    )


    knn.fit(X_train, y_train)
    
    y_pred = knn.predict(X_val)

    f1 = f1_score(y_val, y_pred, average="weighted")

    return f1

study = optuna.create_study(direction="maximize")
study.optimize(objective_knn, n_trials=200)

# print the mean f1 score for the best performing parameter
# print(
#     f"| RBF SVM | {best_results['acc']:.4f} +/- {best_results['acc_std']:.4f} | {best_results['f1_macro']:.4f} +/- {best_results['f1_macro_std']:.4f} | {best_results['f1_weighted']:.4f} +/- {best_results['f1_weighted_std']:.4f} |"
# )
print(f"study {study.best_value=}, {study.best_params=}")

[I 2024-03-15 18:10:10,637] A new study created in memory with name: no-name-7ac9ad89-ef2e-40b8-bc6f-553df3fc9b4f
[I 2024-03-15 18:10:10,641] Trial 0 finished with value: 0.6196969696969696 and parameters: {'n_neighbors': 2, 'weights': 'distance'}. Best is trial 0 with value: 0.6196969696969696.
[I 2024-03-15 18:10:10,643] Trial 1 finished with value: 0.5642857142857144 and parameters: {'n_neighbors': 29, 'weights': 'uniform'}. Best is trial 0 with value: 0.6196969696969696.
[I 2024-03-15 18:10:10,646] Trial 2 finished with value: 0.6215447154471545 and parameters: {'n_neighbors': 18, 'weights': 'uniform'}. Best is trial 2 with value: 0.6215447154471545.
[I 2024-03-15 18:10:10,649] Trial 3 finished with value: 0.5642857142857144 and parameters: {'n_neighbors': 29, 'weights': 'distance'}. Best is trial 2 with value: 0.6215447154471545.
[I 2024-03-15 18:10:10,651] Trial 4 finished with value: 0.6180555555555556 and parameters: {'n_neighbors': 23, 'weights': 'uniform'}. Best is trial 2 wi

study study.best_value=0.6700757575757577, study.best_params={'n_neighbors': 13, 'weights': 'distance'}
knn f1:  0.5289480923909909


In [26]:
knn = KNeighborsClassifier(
    n_neighbors=study.best_params["n_neighbors"],
    weights=study.best_params["weights"]
)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

f1 = f1_score(y_test, y_pred, average="weighted")

print("knn f1: ", f1)

knn f1:  0.5289480923909909


## KNNs
f1 score of ~0.53

In [141]:
C_range: tuple = (1e-3, 1e3)
gamma_range: tuple = (1e-3, 1e3)

# eval on rbf svms
def objective(trial):

    svm = SVC(
        C=trial.suggest_float("C", C_range[0], C_range[1], log=True),
        gamma=trial.suggest_float(
            "gamma", gamma_range[0], gamma_range[1], log=True
        ),
        class_weight=trial.suggest_categorical("class_weight", ["balanced", None]),
        kernel="rbf",
    )

    svm.fit(X_train, y_train)
    
    y_pred = svm.predict(X_val)

    f1 = f1_score(y_val, y_pred, average="weighted")

    return f1

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=200)

# print the mean f1 score for the best performing parameter
# print(
#     f"| RBF SVM | {best_results['acc']:.4f} +/- {best_results['acc_std']:.4f} | {best_results['f1_macro']:.4f} +/- {best_results['f1_macro_std']:.4f} | {best_results['f1_weighted']:.4f} +/- {best_results['f1_weighted_std']:.4f} |"
# )
print(f"study {study.best_value=}, {study.best_params=}")

[I 2024-03-15 15:31:27,643] A new study created in memory with name: no-name-1a3cca76-b5c8-4a71-9ac7-b8566b38ac96
[I 2024-03-15 15:31:27,670] Trial 0 finished with value: 0.2450980392156863 and parameters: {'C': 2.5579742486510035, 'gamma': 328.8687228817511, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.2450980392156863.
[I 2024-03-15 15:31:27,695] Trial 1 finished with value: 0.2450980392156863 and parameters: {'C': 536.6683158456692, 'gamma': 19.2081274433793, 'class_weight': None}. Best is trial 0 with value: 0.2450980392156863.
[I 2024-03-15 15:31:27,718] Trial 2 finished with value: 0.2993827160493827 and parameters: {'C': 3.4224142963574273, 'gamma': 0.00784937662645934, 'class_weight': 'balanced'}. Best is trial 2 with value: 0.2993827160493827.
[I 2024-03-15 15:31:27,743] Trial 3 finished with value: 0.2450980392156863 and parameters: {'C': 6.224230314635548, 'gamma': 233.36947616197656, 'class_weight': 'balanced'}. Best is trial 2 with value: 0.2993827160493827.


study study.best_value=0.8586396546922862, study.best_params={'C': 0.8388442250794123, 'gamma': 0.0024090226780970093, 'class_weight': 'balanced'}


In [142]:
# fit model with best params and test on test set
svm = SVC(
        C=study.best_params["C"],
        gamma=study.best_params["gamma"],
        class_weight=study.best_params["class_weight"],
        kernel="rbf",
)

svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)

f1 = f1_score(y_test, y_pred, average="weighted")

print("svm f1: ", f1)

0.8321924076448073


## SVMs with rbf kernel
- weighted f1 score with ~0.8 on test set
- with normalization wf1 score ~0.83 on test set

In [ ]:
# build mogonet style GNN with
# a. thresholded cosine similarity


In [128]:
def cosine_similarity_matrix(matrix):
    """
    given a matrix of (n_samples, n_features) compute the cosine similarities, between the samples
    """
    # Compute dot product between all pairs of vectors
    dot_products = np.dot(matrix, matrix.T)
    
    # Compute magnitudes of all vectors
    magnitudes = np.linalg.norm(matrix, axis=1)
    
    # Compute outer product of magnitudes to obtain matrix of magnitudes product
    magnitude_products = np.outer(magnitudes, magnitudes)
    
    # Compute cosine similarity matrix
    cosine_similarities = dot_products / magnitude_products
    
    return cosine_similarities

# Example matrix of vectors (rows represent vectors)
matrix = np.array([[1, 1, 1],
                   [2, 2, 2],
                   [1, -1, 1]])

# Compute cosine similarity between all vectors
cos_sim_matrix = cosine_similarity_matrix(matrix)

print("Cosine similarity matrix:")
print(cos_sim_matrix)

Cosine similarity matrix:
[[1.         1.         0.33333333]
 [1.         1.         0.33333333]
 [0.33333333 0.33333333 1.        ]]


In [ ]:
A = cosine_similarity_matrix(X_train)

In [ ]:
# b. moglam style cosine similarity + learned graph

In [ ]:
# build bipartite graph style GNN

In [ ]:
# a. w interactions

In [ ]:
# b. w/o interactions

In [ ]:
# eval mogonet style model


In [ ]:
# eval bipartite style model
